### Colab Instructions

- **Files and Libraries Are Handled Automatically**  
  The dataset, model checkpoint, `compute_cost.py`, and all required libraries will be automatically downloaded and installed when you run the notebook.

- **Enable GPU Acceleration**  
  Go to **Runtime** → **Change runtime type**, and select **GPU** (e.g., **T4 GPU**) as the hardware accelerator.

- **Run the Notebook**  
  Click **Runtime** → **Run all** to execute all cells sequentially.

- **Logging with Weights and Biases**  
  The notebook will prompt you to paste your API token. You can obtain the token by creating a free account at [Weights and Biases](https://wandb.ai/site/).

- **Working Directory**  
  The working directory is set to `/content`.

- **Runtime Duration**  
  Running the full notebook will take approximately 45 minutes.

# Advanced Sound Event Detection Tutorial

In this tutorial, you will learn how to:
- Create a train/validation/test split  
- Evaluate classifiers using standard metrics (e.g., precision, recall, f1-score)  
- Compute segment-level costs based on classifier output  
- Establish a simple baseline
- Train and assess a logistic regression model on audio embeddings  
- Build and evaluate a bidirectional RNN for sequence modeling  
- Compare cost performance across baseline, logistic regression, and RNN models on the test set
- Run inference on the customer's secret test set and store the predictions

In [1]:
# Install required packages
!pip install --quiet numpy pandas matplotlib scikit-learn torch torchvision torchaudio pytorch-lightning wandb rich ipywidgets tabulate tqdm

In [2]:
import os
import pandas as pd
import numpy as np
from tabulate import tabulate
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import pytorch_lightning as pl
from pytorch_lightning.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    LearningRateMonitor,
    RichProgressBar
)
from pytorch_lightning.loggers import WandbLogger
from tqdm import tqdm
from huggingface_hub import snapshot_download, hf_hub_download
import zipfile
import shutil

In [3]:
# download the compute_cost.py file
pyfile_path = hf_hub_download(
    repo_id="fschmid56/mlpc2025_dataset",
    filename="compute_cost.py",
    repo_type="dataset"
)

# move to current working directory (/content)
shutil.copy(pyfile_path, os.getcwd() + "/compute_cost.py")

# import required functions
from compute_cost import CLASSES as TARGET_CLASSES
from compute_cost import COST_MATRIX as TARGET_CLASSES_COST_MATRIX
CLASSES_FP_COSTS = np.array([TARGET_CLASSES_COST_MATRIX[cl]["FP"] for cl in TARGET_CLASSES])
CLASSES_FN_COSTS = np.array([TARGET_CLASSES_COST_MATRIX[cl]["FN"] for cl in TARGET_CLASSES])
print(f"CLASSES_FP_COSTS = {CLASSES_FP_COSTS}")
print(f"CLASSES_FN_COSTS = {CLASSES_FN_COSTS}")
from compute_cost import (
    aggregate_targets,
    get_ground_truth_df,
    get_segment_prediction_df,
    check_dataframe,
    total_cost
)

CLASSES_FP_COSTS = [1 2 3 3 3 3 1 1 3 3]
CLASSES_FN_COSTS = [ 5 10 15 15 15 15  5  5 15 15]


## Download and prepare MLPC2025 Dataset

In [4]:
# Step 1: Download the ZIP file from HF Hub
zip_path = hf_hub_download(
    repo_id="fschmid56/mlpc2025_dataset",   # your dataset repo
    filename="mlpc2025_dataset.zip",        # your uploaded ZIP file
    repo_type="dataset"                     # specify that it's a dataset repo
)
print(f"✅ ZIP downloaded: {zip_path}")

✅ ZIP downloaded: C:\Users\Reinhard\.cache\huggingface\hub\datasets--fschmid56--mlpc2025_dataset\snapshots\5ecbfd8531c18fbb4fa60b79eacdf585b1f1aac4\mlpc2025_dataset.zip


In [5]:
# Step 2: Extract the ZIP
extract_path = "/content/mlpc2025_dataset"
os.makedirs(extract_path, exist_ok=True)

# Check if already extracted
if not os.path.exists(os.path.join(extract_path, "data")):  # assuming 'data/' is inside the zip
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(f"✅ Dataset extracted to {extract_path}")
else:
    print(f"✅ Dataset already extracted at {extract_path}")

✅ Dataset already extracted at /content/mlpc2025_dataset


In [6]:
# Step 3: Set your DATASET_PATH
DATASET_PATH = os.path.join(extract_path, "data")  # because you zipped the 'data' folder
print(f"✅ DATASET_PATH set to {DATASET_PATH}")

# Quick check
print("Files in DATASET_PATH:", os.listdir(DATASET_PATH))

✅ DATASET_PATH set to /content/mlpc2025_dataset\data
Files in DATASET_PATH: ['.cache', 'annotations.csv', 'audio', 'audio_features', 'customer_test_data', 'labels', 'metadata.csv']


In [38]:
METADATA_CSV = os.path.join(DATASET_PATH, 'metadata.csv')
ANNOTATIONS_CSV = os.path.join(DATASET_PATH, 'annotations.csv')
AUDIO_DIR = os.path.join(DATASET_PATH, 'audio')
AUDIO_FEATURES_DIR = os.path.join(DATASET_PATH, 'audio_features')
LABELS_DIR = os.path.join(DATASET_PATH, 'labels')

METADATA = pd.read_csv(METADATA_CSV)
DEV_SET_FILES = METADATA['filename']

CUSTOMER_DATASET_PATH = os.path.join(DATASET_PATH, 'customer_test_data')
CUSTOMER_AUDIO_DIR = os.path.join(CUSTOMER_DATASET_PATH, 'audio')
CUSTOMER_AUDIO_FEATURES_DIR = os.path.join(CUSTOMER_DATASET_PATH, 'audio_features')
CUSTOMER_METADATA_CSV = os.path.join(CUSTOMER_DATASET_PATH, 'metadata.csv')
CUSTOMER_METADATA = pd.read_csv(CUSTOMER_METADATA_CSV)

DATA_SUBSAMPLE = None #3000  # works with available RAM in Colab
NUM_WORKERS = 6  # number of workers (maximum number of workers = CPU cores - 1)
BATCH_SIZE = 128
LOSS_SELECT = 2  # 0 .. nn.BCELoss, 1 .. nn.BCEWithLogitsLoss, 2 .. nn.WeightedBCEWithLogitsLoss

## Create the Data Split

In [39]:
def read_files(file_names, classes, features_dir=AUDIO_FEATURES_DIR, labels_dir=LABELS_DIR):
    """
    Loads features and binary labels for a list of files.

    Returns:
        X: list of np.ndarrays, each of shape (num_frames, num_features)
        Y: dict of lists of np.ndarrays, each of shape (num_frames,)
    """
    X = []
    Y = {c: [] for c in classes} if labels_dir is not None else None

    for fname in file_names:
        base = os.path.splitext(fname)[0]

        # Load features
        feat_path = os.path.join(features_dir, base + '.npz')
        features = np.load(feat_path)['embeddings']  # shape: (T, D)
        X.append(features)

        if labels_dir is not None:
            # Load labels
            label_path = os.path.join(labels_dir, base + '_labels.npz')
            labels = np.load(label_path)

            for c in classes:
                label_array = labels[c]  # shape: (T, num_annotators)
                binary_labels = (np.max(label_array, axis=1) > 0).astype(int)
                Y[c].append(binary_labels)  # shape: (T,)

    return X, Y

In [40]:
# Get filenames for split based on filenames
all_files = DEV_SET_FILES.unique()

# First split: 60% train, 40% temp (val + test)
train_files, temp_files = train_test_split(
    all_files, test_size=0.4, random_state=42, shuffle=True
)

# Second split: 50% val, 50% test from the remaining 40%
val_files, test_files = train_test_split(
    temp_files, test_size=0.5, random_state=42, shuffle=True
)

train_files = train_files[:DATA_SUBSAMPLE]

print(f"Train: {len(train_files)}, Val: {len(val_files)}, Test: {len(test_files)}")

# Load features and labels
X_train, Y_train = read_files(train_files, TARGET_CLASSES)
X_val, Y_val = read_files(val_files, TARGET_CLASSES)
X_test, Y_test = read_files(test_files, TARGET_CLASSES)

Train: 4938, Val: 1646, Test: 1646


In [41]:
print(f"### X ### : ")
print(f"X_train: {type(X_train)}, X_val: {type(X_val)}, X_test: {type(X_test)}")
print(f"X_train[0]: {type(X_train[0])}, X_val[0]: {type(X_val[0])}, X_test[0]: {type(X_test[0])}")
print(f"X_train[0]: {X_train[0].shape}, X_val[0]: {X_val[0].shape}, X_test[0]: {X_test[0].shape}")

print(f"### Y ### : ")
print(f"### Y_train: {len(Y_train)} / {type(Y_train)}")
for y in Y_train :
    print(f"{y} / {type(y)} : {len(Y_train[y])} / {type(Y_train[y])} : {len(Y_train[y][0])} / {type(Y_train[y][0])}")
print(f"### Y_val: {type(Y_val)} / {len(Y_val)}")
for y in Y_val :
    print(f"{y} / {type(y)} : {len(Y_val[y])} / {type(Y_val[y])} : {len(Y_val[y][0])} / {type(Y_val[y][0])}")
print(f"### Y_test: {len(Y_test)} / {type(Y_test)}")
for y in Y_test :
    print(f"{y} / {type(y)} : {len(Y_test[y])} / {type(Y_test[y])} : {len(Y_test[y][0])} / {type(Y_test[y][0])}")

### X ### : 
X_train: <class 'list'>, X_val: <class 'list'>, X_test: <class 'list'>
X_train[0]: <class 'numpy.ndarray'>, X_val[0]: <class 'numpy.ndarray'>, X_test[0]: <class 'numpy.ndarray'>
X_train[0]: (141, 768), X_val[0]: (248, 768), X_test[0]: (200, 768)
### Y ### : 
### Y_train: 10 / <class 'dict'>
Speech / <class 'str'> : 4938 / <class 'list'> : 141 / <class 'numpy.ndarray'>
Shout / <class 'str'> : 4938 / <class 'list'> : 141 / <class 'numpy.ndarray'>
Chainsaw / <class 'str'> : 4938 / <class 'list'> : 141 / <class 'numpy.ndarray'>
Jackhammer / <class 'str'> : 4938 / <class 'list'> : 141 / <class 'numpy.ndarray'>
Lawn Mower / <class 'str'> : 4938 / <class 'list'> : 141 / <class 'numpy.ndarray'>
Power Drill / <class 'str'> : 4938 / <class 'list'> : 141 / <class 'numpy.ndarray'>
Dog Bark / <class 'str'> : 4938 / <class 'list'> : 141 / <class 'numpy.ndarray'>
Rooster Crow / <class 'str'> : 4938 / <class 'list'> : 141 / <class 'numpy.ndarray'>
Horn Honk / <class 'str'> : 4938 / <class

## Evaluation Functions (Metrics & Cost)

In [42]:
# Flatten: Each frame is a sample
def flatten_for_framewise_classification(X, Y_class):
    X_flat = np.concatenate(X)  # shape: (total_frames, num_features)
    Y_flat = np.concatenate(Y_class)  # shape: (total_frames,)
    return X_flat, Y_flat

In [43]:
def evaluate_classifiers(
    classes: list[str],
    Y_val: dict[str, list[np.ndarray]],
    X_val: list[np.ndarray] = None,
    inference_funcs: dict[str, callable] = None,
    Y_pred: dict[str, list[np.ndarray]] = None
) -> tuple[dict[str, list[np.ndarray]], dict[str, dict]]:
    """
    Evaluates per-frame binary classifiers and computes metrics per class.
    Uses either computed predictions or given inference functions.

    Args:
        classes: List of class names to evaluate.
        Y_val: Dict mapping class names to lists of ground-truth (T,) binary arrays.
        X_val: List of input feature arrays, one per validation file. Required if Y_pred not given.
        inference_funcs: Dict mapping class names to binary inference functions.
        Y_pred: Dict with precomputed predictions (same format as Y_val).

    Returns:
        metrics: Dict[class → {'balanced_accuracy', 'precision', 'recall', 'f1'}].
    """

    if Y_pred is None:
        assert inference_funcs is not None and X_val is not None, "If 'Y_pred' is not given, 'inference_funcs' \
                                                                    and 'X_val' must be given."

    Y_val_preds = {}
    metrics     = {}

    for cls in classes:
        # use predictions if given, else infer
        if Y_pred and cls in Y_pred:
            preds_per_file = Y_pred[cls]
        else:
            infer = inference_funcs[cls]
            preds_per_file = [infer(x_file) for x_file in X_val]
        Y_val_preds[cls] = preds_per_file

        # flatten to compute metrics
        y_true = np.concatenate(Y_val[cls])
        y_pred = np.concatenate(preds_per_file)

        metrics[cls] = {
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
            "precision":         precision_score(y_true, y_pred, zero_division=0),
            "recall":            recall_score(y_true, y_pred, zero_division=0),
            "f1":                f1_score(y_true, y_pred, zero_division=0),
        }

    return metrics

In [44]:
def evaluate_cost(
    val_files: list[str],
    dataset_path: str,
    classes: list[str],
    X_val: list[np.ndarray] = None,
    inference_funcs: dict[str, callable] = None,
    Y_pred: dict[str, list[np.ndarray]] = None
):
    """
    Computes segment-level cost based on predictions and ground truth.
    Uses either computed predictions or given inference functions.

    Args:
        val_files: List of filenames corresponding to X_val.
        dataset_path: Path to dataset root (used for loading ground truth).
        classes: List of class names to evaluate.
        X_val: List of input feature arrays, one per validation file. Required if Y_pred not given.
        inference_funcs: Dict mapping class names to binary inference functions.
        Y_pred: Dict with precomputed predictions (class → list of (T,) arrays).

    Returns:
        total: Total cost across all validation files.
        breakdown: Dict[class → segment-level cost].
    """

    if Y_pred is None:
        assert inference_funcs is not None and X_val is not None, "If 'Y_pred' is not given, 'inference_funcs' \
                                                                    and 'X_val' must be given."

    # 0) frame-wise predictions (per class)
    if Y_pred is None:
        Y_pred = {
            cls: [infer(x_file) for x_file in X_val]
            for cls, infer in inference_funcs.items()
        }

    # 1) restructure to filename -> class -> (T,) array
    preds_by_file = {}
    for i, fname in enumerate(val_files):
        preds_by_file[fname] = {
            cls: Y_pred[cls][i] for cls in classes
        }

    # 2) segment-level aggregation using compute_cost
    pred_df = get_segment_prediction_df(
        predictions=preds_by_file,
        class_names=classes
    )

    # 3) load & aggregate ground truth using compute_cost
    gt_df = get_ground_truth_df(val_files, dataset_path)

    # 4) sanity checks from compute_cost
    check_dataframe(pred_df, dataset_path)
    check_dataframe(gt_df, dataset_path)

    # 5) compute cost
    total, breakdown = total_cost(pred_df, gt_df)

    return total, breakdown

## Most-Frequent Label Baseline

In [45]:
def baseline_most_frequent(
    Y_train: dict[str, list[np.ndarray]],
    classes: list[str]
) -> dict[str, callable]:
    """
    Returns inference functions that always predict each class’s majority label.
    """
    inference_funcs = {}
    for cls in classes:
        all_frames = np.concatenate(Y_train[cls])
        most_freq_label  = int(np.mean(all_frames) >= 0.5)
        # inference func ignores features, just returns most frequent label per frame
        inference_funcs[cls] = lambda x, ml=most_freq_label: np.full(x.shape[0], ml, dtype=int)
    return inference_funcs

# 1) Create baseline’s inference functions
bl_inference_funcs = baseline_most_frequent(Y_train, TARGET_CLASSES)

In [46]:
# metrics for most-frequent label baseline
val_metrics = evaluate_classifiers(
    classes=TARGET_CLASSES,
    X_val=X_val,
    Y_val=Y_val,
    inference_funcs=bl_inference_funcs
)

df = pd.DataFrame(val_metrics).T.round(3)
df.columns = ["BAcc", "Precision", "Recall", "F1"]
print(tabulate(df, headers='keys', tablefmt='github'))

|              |   BAcc |   Precision |   Recall |   F1 |
|--------------|--------|-------------|----------|------|
| Speech       |    0.5 |           0 |        0 |    0 |
| Shout        |    0.5 |           0 |        0 |    0 |
| Chainsaw     |    0.5 |           0 |        0 |    0 |
| Jackhammer   |    0.5 |           0 |        0 |    0 |
| Lawn Mower   |    0.5 |           0 |        0 |    0 |
| Power Drill  |    0.5 |           0 |        0 |    0 |
| Dog Bark     |    0.5 |           0 |        0 |    0 |
| Rooster Crow |    0.5 |           0 |        0 |    0 |
| Horn Honk    |    0.5 |           0 |        0 |    0 |
| Siren        |    0.5 |           0 |        0 |    0 |


In [47]:
# cost for most-frequent label baseline
total, breakdown = evaluate_cost(
    val_files=val_files,
    dataset_path=DATASET_PATH,
    classes=TARGET_CLASSES,
    X_val=X_val,
    inference_funcs=bl_inference_funcs
)

df = pd.DataFrame({cls: {"Avg. Cost per minute": round(m["cost"], 4)} for cls, m in breakdown.items()}).T
print(f"Total average cost per minute: {total:.4f}\n")
print(tabulate(df, headers="keys", tablefmt="github"))

Total average cost per minute: 108.8553

|              |   Avg. Cost per minute |
|--------------|------------------------|
| Speech       |                24.8092 |
| Shout        |                 9.4118 |
| Chainsaw     |                 6.6057 |
| Jackhammer   |                 7.4642 |
| Lawn Mower   |                 7.7266 |
| Power Drill  |                14.2369 |
| Dog Bark     |                 7.0986 |
| Rooster Crow |                 0.3816 |
| Horn Honk    |                12.7107 |
| Siren        |                18.4102 |


### Logistic Regression

In [17]:
def train_logistic_regression(
    X_train: list[np.ndarray],
    Y_train: dict[str, list[np.ndarray]],
    classes: list[str]
) -> dict[str, callable]:
    """
    Trains one scaler+logistic-regression per class and returns a dict of
    inference functions. Each function takes a (T, D) feature array and
    returns a (T,) array of {0,1} predictions.
    """
    inference_funcs = {}
    for cls in classes:
        # prepare frame-wise training data
        X_tr, y_tr = flatten_for_framewise_classification(X_train, Y_train[cls])

        # fit scaler and model
        scaler = StandardScaler().fit(X_tr)
        X_tr_scaled = scaler.transform(X_tr)
        clf = LogisticRegression(
            max_iter=100,
            class_weight='balanced',
            random_state=42
        ).fit(X_tr_scaled, y_tr)

        # define and store the joined inference function
        def make_inference(scaler, clf):
            return lambda x: clf.predict(scaler.transform(x))

        inference_funcs[cls] = make_inference(scaler, clf)

    return inference_funcs

lr_inference_funcs = train_logistic_regression(
    X_train, Y_train, TARGET_CLASSES
)

C:\Program Files\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:470: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Program Files\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:470: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please als

In [18]:
val_metrics = evaluate_classifiers(
    classes=TARGET_CLASSES,
    X_val=X_val,
    Y_val=Y_val,
    inference_funcs=lr_inference_funcs
)

df = pd.DataFrame(val_metrics).T.round(3)
df.columns = ["BAcc", "Precision", "Recall", "F1"]
print(tabulate(df, headers='keys', tablefmt='github'))

|              |   BAcc |   Precision |   Recall |    F1 |
|--------------|--------|-------------|----------|-------|
| Speech       |  0.919 |       0.633 |    0.89  | 0.74  |
| Shout        |  0.762 |       0.228 |    0.555 | 0.324 |
| Chainsaw     |  0.899 |       0.535 |    0.805 | 0.643 |
| Jackhammer   |  0.681 |       0.331 |    0.37  | 0.349 |
| Lawn Mower   |  0.752 |       0.387 |    0.513 | 0.441 |
| Power Drill  |  0.735 |       0.224 |    0.502 | 0.31  |
| Dog Bark     |  0.91  |       0.63  |    0.831 | 0.717 |
| Rooster Crow |  0.795 |       0.482 |    0.591 | 0.531 |
| Horn Honk    |  0.784 |       0.272 |    0.591 | 0.373 |
| Siren        |  0.854 |       0.766 |    0.714 | 0.739 |


In [19]:
# inference_funcs from train_logistic_regression_inference(...)
total, breakdown = evaluate_cost(
    val_files=val_files,
    dataset_path=DATASET_PATH,
    classes=TARGET_CLASSES,
    X_val=X_val,
    inference_funcs=lr_inference_funcs
)

df = pd.DataFrame({cls: {"Avg. Cost per minute": round(m["cost"], 4)} for cls, m in breakdown.items()}).T
print(f"Total average cost per minute: {total:.4f}\n")
print(tabulate(df, headers="keys", tablefmt="github"))

Total average cost per minute: 56.3259

|              |   Avg. Cost per minute |
|--------------|------------------------|
| Speech       |                 5.3561 |
| Shout        |                 8.0191 |
| Chainsaw     |                 2.2655 |
| Jackhammer   |                 5.7758 |
| Lawn Mower   |                 5.5469 |
| Power Drill  |                12.3816 |
| Dog Bark     |                 1.6948 |
| Rooster Crow |                 0.2051 |
| Horn Honk    |                 9.601  |
| Siren        |                 5.4801 |


# Bidirectional Gated Recurrent Unit

Can a recurrent neural network, which models temporal dependencies across frames, outperform logistic regression, which treats each frame independently, in sound event detection?

We will implement the required ingredients in the following order:
* Dataset
* DataModule
* RNN Model
* PyTorch Lightning Module
* Hyperparameter Configuration
* Logging via Weights & Biases
* Callbacks
* PyTorch Lightning Trainer

## Dataset



In [48]:
### Exported to SED_Data_Module.py
#class SequenceDataset(Dataset):

from SED_Data_Module import SequenceDataset


In [49]:
ds = SequenceDataset(X_train, Y_train, TARGET_CLASSES, train_files)
feat0, label0, file0 = ds[0]
print("SequenceDataset[0] -> feature shape:", feat0.shape,
      "\nlabel shape:", label0.shape,
      "\nfile[0]:", file0)

SequenceDataset[0] -> feature shape: torch.Size([141, 768]) 
label shape: torch.Size([141, 10]) 
file[0]: 770482.mp3


In [50]:
### Exported to SED_Data_Module.py
# collate_fn used to create batches from the individual dataset items
#def collate_fn(batch):

from SED_Data_Module import collate_fn


In [51]:
batch = [ds[i] for i in range(BATCH_SIZE)]
X_pad, Y_pad, lengths, filenames = collate_fn(batch)

print("collate_fn -> X_padded:", X_pad.shape,
      "\nY_padded:", Y_pad.shape,
      "\nlengths:", lengths,
      "\nfilenames:", filenames[:3], "...")

collate_fn -> X_padded: torch.Size([128, 250, 768]) 
Y_padded: torch.Size([128, 250, 10]) 
lengths: tensor([141, 242, 249, 208, 139, 140, 228, 161, 206, 184, 156, 217, 149, 162,
        152, 128, 183, 127, 137, 234, 187, 166, 212, 239, 214, 128, 143, 159,
        142, 210, 210, 195, 127, 140, 158, 191, 134, 187, 224, 172, 129, 203,
        148, 147, 229, 167, 131, 169, 164, 246, 185, 167, 222, 227, 154, 158,
        144, 210, 164, 145, 156, 218, 244, 221, 205, 157, 178, 152, 235, 167,
        170, 218, 172, 216, 197, 186, 224, 222, 188, 144, 192, 210, 195, 173,
        207, 166, 203, 158, 149, 192, 209, 183, 219, 159, 167, 155, 155, 195,
        133, 133, 145, 214, 202, 195, 141, 188, 214, 249, 169, 164, 133, 204,
        142, 210, 152, 241, 180, 138, 221, 250, 212, 218, 212, 227, 135, 232,
        147, 233]) 
filenames: ['770482.mp3', '649348.mp3', '558402.mp3'] ...


## DataModule

A `LightningDataModule` which organizes **all data loading logic** in one place.

Implements the following core API.

| Method                 | Purpose                                |
|------------------------|----------------------------------------|
| `__init__()`           | Save paths, batch size, classes, etc.  |
| `setup(stage)`         | Prepare datasets (train/val/test)      |
| `train_dataloader()`   | Return DataLoader for training         |
| `val_dataloader()`     | Return DataLoader for validation       |
| `test_dataloader()`    | Return DataLoader for testing          |

In [52]:
### Exported to SED_Data_Module.py
# DataModule is used by pytorch lightning
#class SEDDataModule(pl.LightningDataModule):

from SED_Data_Module import SEDDataModule


In [53]:
dm = SEDDataModule(
    X_train=X_train, Y_train=Y_train, train_files=train_files,
    X_val=X_val,     Y_val=Y_val,     val_files=val_files,
    X_test=X_test,   Y_test=Y_test,   test_files=test_files,
    classes=TARGET_CLASSES,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS
)

dm.setup()
loader = dm.train_dataloader()
X_batch, Y_batch, len_batch, filenames = next(iter(loader))
print("DataModule batch -> X:", X_batch.shape,
      "\nY:", Y_batch.shape,
      "\nlengths:", len_batch,
      "\nfilenames:", filenames[:3], "...")

DataModule batch -> X: torch.Size([128, 250, 768]) 
Y: torch.Size([128, 250, 10]) 
lengths: tensor([149, 215, 222, 139, 173, 176, 132, 215, 218, 195, 210, 193, 181, 209,
        141, 170, 188, 183, 205, 204, 234, 231, 184, 160, 173, 182, 153, 238,
        216, 160, 166, 181, 227, 208, 227, 190, 249, 180, 190, 245, 241, 220,
        180, 238, 187, 136, 143, 214, 223, 143, 221, 159, 218, 134, 131, 169,
        177, 229, 209, 248, 237, 174, 188, 162, 177, 211, 249, 141, 176, 156,
        207, 163, 179, 243, 243, 193, 175, 247, 228, 127, 139, 128, 190, 189,
        221, 243, 243, 199, 155, 222, 178, 128, 136, 200, 180, 234, 152, 146,
        181, 243, 138, 242, 175, 201, 242, 158, 246, 243, 220, 198, 242, 240,
        149, 141, 225, 170, 173, 202, 154, 207, 160, 246, 172, 191, 145, 250,
        163, 171]) 
filenames: ['148297.mp3', '165067.mp3', '640890.mp3'] ...


### Bidirectional RNN

In [54]:
class BiGRUClassifier(nn.Module):
    """
    Bidirectional GRU classifier with a linear output layer.

    Args:
        input_dim: Input feature dimension (D).
        hidden_dim: Hidden size per GRU direction.
        num_layers: Number of stacked GRU layers.
        num_classes: Number of output classes (C).

    Input:
        x: Tensor of shape (B, T, D) — batch of padded sequences.
        lengths: Tensor of shape (B,) — actual lengths before padding.

    Returns:
        logits: Tensor of shape (B, T, C) — class scores for each time step.
    """
    def __init__(self, input_dim, hidden_dim, num_layers, num_classes):
        super().__init__()
        self.gru = nn.GRU(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x, lengths):
        # x: (B, T, D), lengths: (B,)
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_out, _ = self.gru(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        # out: (B, T, 2*hidden_dim)
        logits = self.classifier(out)  # (B, T, num_classes)
        return logits

In [55]:
# Instantiate model
model = BiGRUClassifier(
    input_dim=X_batch.shape[-1],
    hidden_dim=1024,
    num_layers=4, # 2, # TESTING
    num_classes=Y_batch.shape[-1]
)

# Forward pass
logits = model(X_batch, len_batch)

# Print shapes
print("Input X_batch shape:", X_batch.shape)       # (B, T_max, F)
print("Output logits shape:", logits.shape)         # (B, T_max, C)

Input X_batch shape: torch.Size([128, 250, 768])
Output logits shape: torch.Size([128, 250, 10])


In [56]:
# Custom Loss Function for incorporating class specific costs

class WeightedBCEWithLogitsLoss(nn.Module):
    def __init__(self, fp_cost, fn_cost, reduction: str = 'mean'):
        """
        Custom BCEWithLogitsLoss that uses different costs for false positives and false negatives per class.

        Args:
            fp_cost (Tensor): 1D tensor of shape (num_classes,) with cost for false positives.
            fn_cost (Tensor): 1D tensor of shape (num_classes,) with cost for false negatives.
            reduction (str): 'none' | 'mean' | 'sum'
        """
        super().__init__()
        assert reduction in ('none', 'mean', 'sum'), "Invalid reduction type"
        self.register_buffer('fp_cost', torch.tensor(fp_cost, dtype=torch.float32))
        self.register_buffer('fn_cost', torch.tensor(fn_cost, dtype=torch.float32))

        self.reduction = reduction

    def forward(self, input: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """
        Args:
            input (Tensor): Raw logits of shape (batch_size, num_classes)
            target (Tensor): Binary labels of shape (batch_size, num_classes)
        Returns:
            loss (Tensor): Loss scalar or tensor depending on reduction.
        """
        # Apply sigmoid to logits
        #probs = torch.sigmoid(input)

        # Compute weights: FP weight where target==0, FN weight where target==1
        # Shape: (batch_size, num_classes)
        weights = torch.where(target == 1, self.fn_cost, self.fp_cost)

        # Compute BCE loss per element
        bce = F.binary_cross_entropy_with_logits(input, target, reduction='none')

        # Apply per-class weight
        weighted_loss = bce * weights

        if self.reduction == 'mean':
            return weighted_loss.mean()
        elif self.reduction == 'sum':
            return weighted_loss.sum()
        else:  # 'none'
            return weighted_loss

        #criterion = torch.nn.BCEWithLogitsLoss(reduction=self.reduction, pos_weight=weights)
        #return criterion(input, target)

### PyTorch Lightning Module: `SEDLightningModule`

The `LightningModule` wraps your model and training logic, abstracting away boilerplate code and handling key training steps automatically.  
It implements a **standardized API** to define how your model should behave during training, validation, testing, and prediction.

- **`__init__`**: initializes the model (`BiGRUClassifier`), loss function, and validation and test buffer storage, and sets important attributes (e.g., lr, threshold).
- **`forward(x, lengths)`**: forward pass through the GRU model.
- **`predict_step(batch, batch_idx)`**: applies sigmoid + thresholding, slices off padding → returns predictions.
- **`training_step(batch, batch_idx)`**: handles training logic.
- **`validation_step(batch, batch_idx)`**: handles validation logic.
- **`on_validation_epoch_end()`**: aggregates validation results after each epoch.
- **`configure_optimizers()`**: defines the optimizer (Adam).

In [57]:
class SEDLightningModule(pl.LightningModule):
    def __init__(self, input_dim, hidden_dim, num_layers, classes, classes_fp_costs, classes_fn_costs, lr=1e-4, threshold=0.5):
        super().__init__()
        # Core model
        self.model = BiGRUClassifier(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            num_classes=len(classes)
        )

        self.classes = classes
        self.classes_fp_costs = classes_fp_costs
        self.classes_fn_costs = classes_fn_costs

        if 1 == LOSS_SELECT :
            # Loss (we'll apply masking later, thus reduction='none')
            self.criterion = nn.BCEWithLogitsLoss(reduction='none')
        elif 2 == LOSS_SELECT :
            # Custom Loss for class specific costs
            self.criterion = WeightedBCEWithLogitsLoss(fp_cost=self.classes_fp_costs, fn_cost=self.classes_fn_costs, reduction='none')
        self.lr = lr
        self.threshold = threshold

        self._val_preds   = {c: [] for c in self.classes}
        self._val_targets = {c: [] for c in self.classes}
        self._val_filenames = []

    def forward(self, x, lengths):
        return self.model(x, lengths)

    def predict_step(self, batch, batch_idx):
        # unpack batch (with or without labels)
        if len(batch) == 4:
            X, _, lengths, filenames = batch
        else:
            X, lengths, filenames = batch

        # 1) raw logits → probs → binary preds
        logits = self.model(X, lengths)
        probs  = torch.sigmoid(logits)
        preds  = (probs > self.threshold).int() # (B, T_max, C)

        # 2) remove padding
        batch_preds = [preds[b, :lengths[b]].cpu()
                      for b in range(X.size(0))]

        return {"filenames": filenames, "preds": batch_preds}

    # we will implement the processing steps one after the other in the following
    def training_step(self, batch, batch_idx):
        return self.process_training_step(batch, batch_idx)

    def validation_step(self, batch, batch_idx):
        return self.process_validation_step(batch, batch_idx)

    def on_validation_epoch_end(self):
        return self.process_validation_epoch_end()

    def test_step(self, batch, batch_idx):
        return self.process_test_step(batch, batch_idx)

    def on_test_epoch_end(self):
        return self.process_test_epoch_end()

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

### PyTorch Lightning Module: `SEDLightningModule`

In the following, we will implement the missing functions:

- `process_training_step`:  
  computes masked BCE loss for each frame and logs the training loss.

- `process_validation_step`:  
  collects per-frame predictions and targets for later metric computation.

- `process_validation_epoch_end`:  
  aggregates predictions and targets, computes metrics, and logs results.

- `process_test_step`:  
  same as validation but used for test-time evaluation.

- `process_test_epoch_end`:  
  evaluates and logs performance after test epoch.

We will bind this functions to our `SEDLightningModule`.

### `process_training_step`

computes masked BCE loss for each frame and logs the training loss

In [58]:
def process_training_step(self, batch, batch_idx):
    X, Y, lengths, _ = batch      # X: (B, T, D), Y: (B, T, C), lengths: (B,)
    logits = self(X, lengths)     # calls self.forward, results in logits of shape (B, T, C)

    # raw per-element loss
    loss_raw = self.criterion(logits, Y.float())  # (B, T, C)

    # build mask to zero out padded frames
    mask = torch.arange(logits.size(1), device=logits.device)[None, :] < lengths[:, None]
    mask = mask.unsqueeze(-1).float()     # (B, T, 1)

    # apply mask and average
    loss = (loss_raw * mask).sum() / mask.sum()

    self.log('train/loss', loss, prog_bar=True, on_step=True, on_epoch=True, batch_size=X.size(0))
    return loss

# Bind it to the LightningModule
SEDLightningModule.process_training_step = process_training_step

### `process_validation_step`

computes masked BCE loss, logs it, and stores frame-level predictions and targets for aggreation in `process_validation_epoch_end`

In [59]:
def process_validation_step(self, batch, batch_idx):
    X, Y, lengths, filenames = batch      # X: (B, T, D), Y: (B, T, C), lengths: (B,)
    logits = self(X, lengths)             # calls self.forward, results in logits of shape (B, T, C)

    # Determine logging prefix
    prefix = "test" if self.trainer.testing else "val"

    # compute masked BCE loss
    loss_raw = self.criterion(logits, Y.float())     # (B, T, C)
    mask = torch.arange(logits.size(1), device=logits.device)[None, :] < lengths[:, None]
    mask = mask.unsqueeze(-1).float()                # (B, T, 1)
    loss = (loss_raw * mask).sum() / mask.sum()

    self.log(f'{prefix}/loss', loss, prog_bar=True, on_step=False, on_epoch=True, batch_size=X.size(0))

    # store frame-wise preds & targets for epoch_end
    # frame-wise logits are thresholded here
    preds = (torch.sigmoid(logits) > self.threshold).long()     # (B, T, C)
    self._val_filenames.extend(filenames)

    for i, c in enumerate(self.classes):
        for b in range(X.size(0)):
            T = lengths[b]
            self._val_preds[c].append(preds[b, :T, i])
            self._val_targets[c].append(Y[b, :T, i])

    return loss

# Bind it to the LightningModule
SEDLightningModule.process_validation_step = process_validation_step

### `process_validation_epoch_end`

computes dataset metrics and cost and logs them at the end of a validation epoch.

In [60]:
def process_validation_epoch_end(self):
    # Determine current mode
    prefix = "test" if self.trainer.testing else "val"

    # --- 1) Convert buffered tensors to NumPy arrays ---
    preds_numpy = {
        cls: [p.cpu().numpy() for p in self._val_preds[cls]]
        for cls in self.classes
    }
    targets_numpy = {
        cls: [t.cpu().numpy() for t in self._val_targets[cls]]
        for cls in self.classes
    }

    # --- 2) Frame‐level metrics ---
    frame_metrics = evaluate_classifiers(
        classes=self.classes,
        Y_val=targets_numpy,
        Y_pred=preds_numpy
    )

    for cls, m in frame_metrics.items():
        self.log(f'{prefix}/{cls}_bacc',     m['balanced_accuracy'])
        self.log(f'{prefix}/{cls}_precision',m['precision'])
        self.log(f'{prefix}/{cls}_recall',   m['recall'])
        self.log(f'{prefix}/{cls}_f1',       m['f1'])

    # --- 3) Segment‐level cost ---
    total_cost, cost_breakdown = evaluate_cost(
        val_files=self._val_filenames,
        dataset_path=DATASET_PATH,
        classes=self.classes,
        Y_pred=preds_numpy
    )
    self.log(f'{prefix}/total_cost', total_cost, prog_bar=True)
    for cls, cls_cost in cost_breakdown.items():
        self.log(f"{prefix}/cost/{cls}", cls_cost["cost"], prog_bar=False)

    # --- 4) Clear buffers ---
    self._val_preds     = {c: [] for c in self.classes}
    self._val_targets   = {c: [] for c in self.classes}
    self._val_filenames = []


SEDLightningModule.process_validation_epoch_end = process_validation_epoch_end

### `process_test_step` and `process_test_epoch_end` ...

fortunately require the same logic as validation, so we can reuse `process_validation_step` and `process_validation_epoch_end`

In [61]:
# After you’ve attached the validation logic, simply reuse it for testing:

# Reuse the same step‐logic
SEDLightningModule.process_test_step = SEDLightningModule.process_validation_step

# Reuse the same epoch‐end logic
SEDLightningModule.process_test_epoch_end = SEDLightningModule.process_validation_epoch_end

### Hyperparameters

Key hyperparameters with reasonable initial values — most likely not guaranteed optimal.

In [62]:
hparams = dict(
    # not tuned by us - used out of the box
    input_dim      = X_batch.shape[-1],
    hidden_dim     = 1024,
    num_layers     = 4,
    lr             = 1e-4,
    batch_size     = BATCH_SIZE,
    max_epochs     = 50,
    threshold      = 0.5,
    patience       = 5,         # Early-stopping patience
)

### Callbacks

Callbacks are modular hooks that enable custom actions during training (e.g., saving checkpoints, early stopping, or logging), triggered at specific stages.

In [63]:
checkpoint_cb = ModelCheckpoint(
    monitor    = "val/total_cost",   # minimize cost
    mode       = "min",
    save_top_k = 1,                  # save top model on validation data
    filename   = "best-{epoch:02d}"
)

early_stop_cb = EarlyStopping(
    monitor  = "val/total_cost",
    mode     = "min",
    patience = hparams["patience"],
    verbose  = True
)

lr_monitor_cb = LearningRateMonitor(logging_interval="epoch")

# RichProgressBar generates minimal output compared to 'tqdm'
progress_bar_cb = RichProgressBar()

callbacks = [checkpoint_cb, early_stop_cb, lr_monitor_cb, progress_bar_cb]

### Logger

- [Weights & Biases (wandb)](https://wandb.ai/site/) is a powerful and free experiment tracking tool  
- lets you log metrics, visualize training runs, compare models  
- share results via an interactive online dashboard  
- integrates seamlessly with PyTorch Lightning

In [64]:
wandb_logger = WandbLogger(
    project     = "mlpc2025-sed",
    name        = f"BiGRU-{hparams['hidden_dim']}x{hparams['num_layers']}",
    config      = hparams
)

### Trainer

The `Trainer` is the central PyTorch Lightning component that orchestrates training, validation, and testing.

It brings everything together:
- The `SEDDataModule` provides the data.
- The `SEDLightningModule` defines the model and training logic.
- The `Trainer` handles the training loop, evaluation, logging, and callbacks.

In [66]:
dm = SEDDataModule(
    X_train=X_train, Y_train=Y_train, train_files=train_files,
    X_val=X_val,     Y_val=Y_val,     val_files=val_files,
    X_test=X_test,   Y_test=Y_test,   test_files=test_files,
    classes=TARGET_CLASSES,
    batch_size=hparams["batch_size"],
    num_workers=NUM_WORKERS
)

model = SEDLightningModule(
    input_dim  = hparams["input_dim"],
    hidden_dim = hparams["hidden_dim"],
    num_layers = hparams["num_layers"],
    classes    = TARGET_CLASSES,
    classes_fp_costs = CLASSES_FP_COSTS,
    classes_fn_costs = CLASSES_FN_COSTS,
    lr         = hparams["lr"]
)

trainer = pl.Trainer(
    accelerator             = "gpu",
    devices                 = 1,
    max_epochs              = hparams["max_epochs"],
    callbacks               = callbacks,
    logger                  = wandb_logger,
    log_every_n_steps       = 10,
    deterministic           = True,
    check_val_every_n_epoch = 1,
    num_sanity_val_steps    = 0
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


### Let's train!

The following command launches training and validation, alternating across epochs. All results will be logged to Weights & Biases.
You can explore a completed training run here: https://api.wandb.ai/links/cp_tobi/plk26iu9

Checkpoints will stored in `mlpc2025-sed/<wandb_id>/checkpoints`.

In [67]:
###
#ckpt_path = ".\wandb\run-20250608_184502-dgwmepda"
#Wandb.init() # login --relogin # from CMD

trainer.fit(model, datamodule=dm, )   # train and validate

You are using a CUDA device ('NVIDIA RTX A5000 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
wandb: Currently logged in as: ki-8-technik (ki-8-technik-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type                      ┃ Params ┃ Mode  ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ BiGRUClassifier           │ 67.7 M │ train │
│ 1 │ criterion │ WeightedBCEWithLogitsLoss │      0 │ train │
└───┴───────────┴───────────────────────────┴────────┴───────┘

Trainable params: 67.7 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 67.7 M                                                                                               
Total estimated model params size (MB): 270                                                                        
Modules in train mode: 4                                                                                           
Modules in eval mode: 0

C:\Program Files\Python312\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:420: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.
C:\Program Files\Python312\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:420: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.


Output()

Metric val/total_cost improved. New best score: 49.547
Metric val/total_cost improved by 6.289 >= min_delta = 0.0. New best score: 43.258
Metric val/total_cost improved by 2.114 >= min_delta = 0.0. New best score: 41.143
Metric val/total_cost improved by 0.329 >= min_delta = 0.0. New best score: 40.814
Metric val/total_cost improved by 0.766 >= min_delta = 0.0. New best score: 40.048
Monitored metric val/total_cost did not improve in the last 5 records. Best score: 40.048. Signaling Trainer to stop.


### Let's test!

This loads the checkpoint with the lowest validation cost and runs evaluation on the test set.
Check example test results logged to Weights & Biases here: https://api.wandb.ai/links/cp_tobi/plk26iu9

In [68]:
test_results = trainer.test(model, datamodule=dm, ckpt_path="best")   # test

Restoring states from the checkpoint path at .\mlpc2025-sed\cpqw2wpy\checkpoints\best-epoch=06.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at .\mlpc2025-sed\cpqw2wpy\checkpoints\best-epoch=06.ckpt
C:\Program Files\Python312\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:420: Consider setting `persistent_workers=True` in 'test_dataloader' to speed up the dataloader worker initialization.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃         Test metric         ┃        DataLoader 0         ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│     test/Chainsaw_bacc      │     0.8894338607788086      │
│      test/Chainsaw_f1       │     0.8173097372055054      │
│   test/Chainsaw_precision   │     0.8584863543510437      │
│    test/Chainsaw_recall     │     0.7799023389816284      │
│     test/Dog Bark_bacc      │     0.8927432298660278      │
│      test/Dog Bark_f1       │     0.7096635103225708      │
│   test/Dog Bark_precision   │     0.6410887837409973      │
│    test/Dog Bark_recall     │     0.7946658134460449      │
│     test/Horn Honk_bacc     │     0.7874148488044739      │
│      test/Horn Honk_f1      │     0.45646941661834717     │
│  test/Horn Honk_precision   │     0.37217485904693604     │
│    test/Horn Honk_recall    │     0.5901287794113159      │
│    test/Jackhammer_bacc     │     0.7673762440681458      │
│     test/Jackhammer_f1      │     0.42052605748176575     │
│  test/Jackhammer_precision  │     0.34316566586494446     │
│   test/Jackhammer_recall    │     0.5429166555404663      │
│    test/Lawn Mower_bacc     │     0.6996370553970337      │
│     test/Lawn Mower_f1      │     0.5196428298950195      │
│  test/Lawn Mower_precision  │     0.7404580116271973      │
│   test/Lawn Mower_recall    │     0.4002751111984253      │
│    test/Power Drill_bacc    │      0.823453426361084      │
│     test/Power Drill_f1     │     0.3353920578956604      │
│ test/Power Drill_precision  │     0.2235306352376938      │
│   test/Power Drill_recall   │      0.67136150598526       │
│   test/Rooster Crow_bacc    │     0.7995526790618896      │
│    test/Rooster Crow_f1     │     0.7393617033958435      │
│ test/Rooster Crow_precision │     0.9652777910232544      │
│  test/Rooster Crow_recall   │     0.5991379022598267      │
│       test/Shout_bacc       │     0.7728712558746338      │
│        test/Shout_f1        │     0.4403630197048187      │
│    test/Shout_precision     │     0.3613475263118744      │
│      test/Shout_recall      │     0.5636062026023865      │
│       test/Siren_bacc       │     0.9129660129547119      │
│        test/Siren_f1        │     0.7679324746131897      │
│    test/Siren_precision     │     0.7112596035003662      │
│      test/Siren_recall      │     0.8344185948371887      │
│      test/Speech_bacc       │     0.9161585569381714      │
│       test/Speech_f1        │     0.7533105611801147      │
│    test/Speech_precision    │      0.656132698059082      │
│     test/Speech_recall      │     0.8842783570289612      │
│     test/cost/Chainsaw      │      1.461533546447754      │
│     test/cost/Dog Bark      │     1.3552113771438599      │
│     test/cost/Horn Honk     │      6.941094398498535      │
│    test/cost/Jackhammer     │      3.875206232070923      │
│    test/cost/Lawn Mower     │      3.403897523880005      │
│    test/cost/Power Drill    │      6.222229480743408      │
│   test/cost/Rooster Crow    │     0.15710295736789703     │
│       test/cost/Shout       │      6.065126419067383      │
│       test/cost/Siren       │     4.4274468421936035      │
│      test/cost/Speech       │     5.1891584396362305      │
│          test/loss          │     2.4152188301086426      │
│       test/total_cost       │      39.09800720214844      │
└─────────────────────────────┴─────────────────────────────┘

## Compare Baseline, Logistic Regression and BiGRU Costs on Test Set

In [69]:
# baseline inference on test set
bl_total, bl_breakdown = evaluate_cost(
    test_files,
    DATASET_PATH,
    TARGET_CLASSES,
    X_test,
    bl_inference_funcs
)

In [70]:
# logistic regression inference on test set
lr_total, lr_breakdown = evaluate_cost(
    test_files,
    DATASET_PATH,
    TARGET_CLASSES,
    X_test,
    lr_inference_funcs
)

### Collect all costs in a pandas dataframe for pretty print

In [71]:
# shuffle around format for pretty print

# Convert breakdowns into dict[class → cost]
bl_costs = {cls: d["cost"] for cls, d in bl_breakdown.items()}
lr_costs = {cls: d["cost"] for cls, d in lr_breakdown.items()}

# Add total cost
bl_costs["TOTAL"] = bl_total
lr_costs["TOTAL"] = lr_total

# Extract relevant costs from pytorch lightning test results
rnn_result = test_results[0]
rnn_costs = {
    cls: rnn_result[f"test/cost/{cls}"]
    for cls in TARGET_CLASSES
    if f"test/cost/{cls}" in rnn_result
}

# Add total cost
rnn_costs["TOTAL"] = rnn_result["test/total_cost"]

# Create a DataFrame for comparison
cost_df = pd.DataFrame({
    "Baseline": bl_costs,
    "Logistic Regression": lr_costs,
    "RNN": rnn_costs
}).round(2)

In [72]:
print(tabulate(cost_df.reset_index().values,
               headers=["Class", "Baseline", "Logistic Regression", "RNN"],
               tablefmt="github"))

| Class        |   Baseline |   Logistic Regression |   RNN |
|--------------|------------|-----------------------|-------|
| Speech       |      27.05 |                  6.17 |  5.19 |
| Shout        |       9.92 |                  8.74 |  6.07 |
| Chainsaw     |       6.05 |                  2.07 |  1.46 |
| Jackhammer   |       6.36 |                  5.98 |  3.88 |
| Lawn Mower   |       5.36 |                  4.02 |  3.4  |
| Power Drill  |       8.24 |                  9.33 |  6.22 |
| Dog Bark     |       6.16 |                  2.04 |  1.36 |
| Rooster Crow |       0.48 |                  0.19 |  0.16 |
| Horn Honk    |      13.12 |                  9.18 |  6.94 |
| Siren        |      18.76 |                  5.97 |  4.43 |
| TOTAL        |     101.47 |                 53.68 | 39.1  |


## Compute predictions on customer's secret test set

Requires three functions:
- `load_model_from_checkpoint`: loading the desired model checkpoint, checkpoints are in folder `mlpc2025-sed/<wandb_id>/checkpoints`; an example checkpoint will be downloaded below
- `predict_dataset`: generate predictions for customer's dataset
- `segment_and_save`: bring predictions into the required 1.2 second segement format and save as csv file

In [79]:
def load_model_from_checkpoint(
    ckpt_path: str,
    hparams: dict,
    classes: list[str]
) -> pl.LightningModule:
    return SEDLightningModule.load_from_checkpoint(
        checkpoint_path=ckpt_path,
        input_dim  = hparams["input_dim"],
        hidden_dim = hparams["hidden_dim"],
        num_layers = hparams["num_layers"],
        lr         = hparams["lr"],
        threshold  = hparams["threshold"],
        classes    = classes,
        classes_fp_costs = CLASSES_FP_COSTS,
        classes_fn_costs = CLASSES_FN_COSTS,
    )

In [80]:
def predict_dataset(
    model: pl.LightningModule,
    loader: DataLoader
) -> dict[str, dict[str, np.ndarray]]:
    """
    Runs trainer.predict() on `loader` and returns:
      preds_by_file[filename][class] = 1D NumPy array of frame‐wise {0,1}.
    """
    trainer = pl.Trainer(accelerator="gpu", devices=1) # accelerator="auto"
    outputs = trainer.predict(model, dataloaders=loader)

    # flatten into lists
    all_preds = {c: [] for c in model.classes}
    all_files = []
    for batch_out in outputs:
        for fname, pred in zip(batch_out["filenames"], batch_out["preds"]):
            all_files.append(fname)
            arr = pred.numpy()  # shape (T_i, C)
            for i, cls in enumerate(model.classes):
                all_preds[cls].append(arr[:, i])

    # repackage into preds_by_file
    preds_by_file: dict[str, dict[str, np.ndarray]] = {}
    for idx, fname in enumerate(all_files):
        preds_by_file.setdefault(fname, {})
        for cls in model.classes:
            preds_by_file[fname][cls] = all_preds[cls][idx]

    return preds_by_file

In [81]:
def segment_and_save(
    preds_by_file: dict[str, dict[str, np.ndarray]],
    class_names: list[str],
    dataset_path: str,
    out_csv: str,
    compute_cost: bool = False,
    test_files: list[str] = None,
) -> pd.DataFrame:
    """
    1) Build segment‐level DataFrame
    2) Sanity‐check with check_dataframe()
    3) (optional) compute & print cost if val_files is provided
    4) save CSV to out_csv
    """
    # 1) aggregate predictions using the function provided in compute_cost.py
    pred_df = get_segment_prediction_df(
        predictions = preds_by_file,
        class_names = class_names
    )

    # 2) sanity‐check (from compute_cost.py)
    check_dataframe(pred_df, dataset_path)

    # 3) cost (optional), for sanity check on our custom test split
    if compute_cost and test_files is not None:
        gt_df = get_ground_truth_df(test_files, dataset_path) # from compute_cost.py
        total, breakdown = total_cost(pred_df, gt_df) # from compute_cost.py
        print(f"\nTotal cost: {total:.4f}")

        gt_csv = os.path.splitext(out_csv)[0] + "_ground_truth.csv"
        gt_df.to_csv(gt_csv, index=False)
        print(f"Saved ground truth segments to {gt_csv}")

    # 4) save
    pred_df.to_csv(out_csv, index=False)
    print(f"Saved segment predictions to {out_csv}")

    return pred_df

### Load checkpoint

Download an example checkpoint from huggingface.

In [82]:
# download example checkpoint from huggingface
ckpt_path = hf_hub_download(
    repo_id="fschmid56/mlpc2025_dataset",
    filename="colab_tutorial.ckpt",
    repo_type="model"
)

# alternatively, use your own local checkpoint
# replace wandb id 'lo9ygyg4' with your desired wandb id
# replace 'best-epoch=11.ckpt' with the name of your checkpoint
# ckpt_path = "/content/mlpc2025-sed/lo9ygyg4/checkpoints/best-epoch=11.ckpt"

ckpt_path = "./mlpc2025-sed/cpqw2wpy/checkpoints/best-epoch=06.ckpt" # ".\mlpc2025-sed\dgwmepda\checkpoints\best-epoch=04.ckpt"

model = load_model_from_checkpoint(ckpt_path, hparams, TARGET_CLASSES)

We sanity check model loading, the prediction routine and segmenting predictions by applying it our custom test split and calculating costs (as we have access to the labels).

In [83]:
# 1) TEST SPLIT
test_dataset = SequenceDataset(X_test, Y_test, TARGET_CLASSES, test_files)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)
test_preds   = predict_dataset(model, test_loader)
segment_and_save(
    preds_by_file = test_preds,
    class_names   = TARGET_CLASSES,
    dataset_path  = DATASET_PATH,
    out_csv       = "test_split_predictions.csv",
    compute_cost  = True,
    test_files     = test_files,
)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
C:\Program Files\Python312\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …


Total cost: 39.0980
Saved ground truth segments to test_split_predictions_ground_truth.csv
Saved segment predictions to test_split_predictions.csv


,filename,onset,Speech,Shout,Chainsaw,Jackhammer,Lawn Mower,Power Drill,Dog Bark,Rooster Crow,Horn Honk,Siren
0,440698.mp3,0.0,0,0,0,0,0,0,0,0,0,0
1,440698.mp3,1.2,0,0,0,0,0,0,0,0,0,0
2,440698.mp3,2.4,0,0,0,0,0,0,0,0,0,0
3,440698.mp3,3.6,0,0,0,0,0,0,0,0,0,0
4,440698.mp3,4.8,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
31503,400159.mp3,19.2,0,0,0,0,0,0,0,0,0,0
31504,400159.mp3,20.4,0,0,0,0,0,0,0,0,0,0
31505,400159.mp3,21.6,0,0,0,0,0,0,0,0,0,0
31506,400159.mp3,22.8,0,0,0,0,0,0,0,0,0,0


Finally, compute predictions on the customer's secret test set and store as `/content/customer_predictions.csv`.

In [84]:
# 2) CUSTOMER SET (no labels → compute_cost=False)
customer_files = CUSTOMER_METADATA["filename"].unique()
X_cust, _ = read_files(customer_files, TARGET_CLASSES,
                       features_dir=CUSTOMER_AUDIO_FEATURES_DIR,
                       labels_dir=None)
cust_dataset = SequenceDataset(X_cust, None, TARGET_CLASSES, customer_files)
cust_loader  = DataLoader(cust_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)

cust_preds = predict_dataset(model, cust_loader)
segment_and_save(
    preds_by_file = cust_preds,
    class_names   = TARGET_CLASSES,
    dataset_path  = CUSTOMER_DATASET_PATH,
    out_csv       = "customer_predictions.csv",
    compute_cost  = False,  # can't compute on customer's secret test set
)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
C:\Program Files\Python312\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

Saved segment predictions to customer_predictions.csv


,filename,onset,Speech,Shout,Chainsaw,Jackhammer,Lawn Mower,Power Drill,Dog Bark,Rooster Crow,Horn Honk,Siren
0,386984.mp3,0.0,1,0,0,0,0,0,0,0,0,0
1,386984.mp3,1.2,1,0,0,0,0,0,0,0,0,0
2,386984.mp3,2.4,1,0,0,0,0,0,0,0,0,0
3,386984.mp3,3.6,1,0,0,0,0,0,0,0,0,0
4,386984.mp3,4.8,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
52186,507531.mp3,21.6,0,0,0,0,0,0,0,0,0,0
52187,507531.mp3,22.8,0,0,0,0,0,0,0,0,0,0
52188,507531.mp3,24.0,0,0,0,0,0,0,0,0,0,0
52189,507531.mp3,25.2,0,0,0,0,0,0,0,0,0,0


### Final checks as in Task Description

Instead of importing all the functions from `compute_cost.py` and working with DataFrames directly, you can also run the provided script as recommended in the Task Description. Just pass your generated .csv file(s) to verify correctness and compute cost.

In [85]:
!python compute_cost.py \
  --dataset_path="{DATASET_PATH}" \
  --ground_truth_csv="test_split_predictions_ground_truth.csv" \
  --predictions_csv="test_split_predictions.csv"

Predictions CSV formated correctly.
Ground truth CSV formated correctly.
Total cost: 39.0980068554018


In [86]:
!python compute_cost.py \
  --dataset_path="{CUSTOMER_DATASET_PATH}" \
  --predictions_csv="customer_predictions.csv"

Predictions CSV formated correctly.
